In [31]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [ ]:
!pip install mlflow dagshub
import pandas as pd
import numpy as np
import mlflow
import dagshub
import category_encoders as ce
import gc
import random
import copy
import shap
import xgboost as xgb
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler

# Train/Test Split

In [2]:
dagshub_username = "nikaduri"
repo_name = "ml-ieee-cis-fraud-detection"
dagshub.init(repo_owner=dagshub_username, repo_name=repo_name, mlflow=True)

# Separate experiment for XGBoost Architecture
mlflow.set_experiment("XGBoost_Training")

print("Loading data...")
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

df = train_transaction.merge(train_identity, on='TransactionID', how='left')
del train_transaction, train_identity 
gc.collect()

X = df.drop(columns=['isFraud'])
y = df['isFraud']

# First Split: Carve out the 20% Test Set
X_temp, X_test_raw, y_temp, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Second Split: Split the remaining 80% into Train (60%) and Validation (20%)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, shuffle=False) 

del df, X, y, X_temp, y_temp 
gc.collect()

print(f"Data Splits - Train: {X_train_raw.shape[0]} | Val: {X_val_raw.shape[0]} | Test: {X_test_raw.shape[0]}")

Data Splits - Train: 354324 | Val: 118108 | Test: 118108


# Feature Cleaning

In [4]:
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    print("\n--- Executing Cleaning Stage ---")
    nan_threshold = 0.90
    mlflow.log_param("nan_drop_threshold", nan_threshold)
    
    irrelevant_cols = ['TransactionID', 'TransactionDT']
    
    missing_fractions = X_train_raw.isnull().mean()
    cols_to_keep_nan = missing_fractions[missing_fractions <= nan_threshold].index.tolist()
    cleaned_features = [c for c in cols_to_keep_nan if c not in irrelevant_cols]
    
    mlflow.log_metric("features_after_cleaning", len(cleaned_features))
    
    del missing_fractions
    gc.collect()


--- Executing Cleaning Stage ---
🏃 View run XGBoost_Cleaning at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/8/runs/f52569d0e8f64bdf8595f61b972b986b
🧪 View experiment at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/8


# Feature Engineering

In [5]:
with mlflow.start_run(run_name="XGBoost_SHAP_Feature_Selection"):
    shap_threshold = 0.001
    mlflow.log_param("shap_threshold", shap_threshold)
    
    # 1. Temporary preprocessing strictly for SHAP calculation
    X_train_temp = X_train_raw[cleaned_features].copy()
    cat_cols_temp = X_train_temp.select_dtypes(include=['object']).columns.tolist()
    num_cols_temp = X_train_temp.select_dtypes(exclude=['object']).columns.tolist()
    
    X_train_temp[cat_cols_temp] = X_train_temp[cat_cols_temp].fillna('Missing_Category')
    X_train_temp[num_cols_temp] = X_train_temp[num_cols_temp].fillna(X_train_temp[num_cols_temp].median())
    
    woe = ce.WOEEncoder(cols=cat_cols_temp)
    X_train_encoded = woe.fit_transform(X_train_temp, y_train)
    
    del X_train_temp 
    gc.collect()
    
    print("Training baseline XGBoost for SHAP extraction...")
    baseline_xgb = xgb.XGBClassifier(
        n_estimators=100, max_depth=5, scale_pos_weight=2.33, 
        random_state=67, n_jobs=-1, eval_metric="auc"
    )
    baseline_xgb.fit(X_train_encoded, y_train)
    
    explainer = shap.TreeExplainer(baseline_xgb)
    X_sample = X_train_encoded.sample(n=10000, random_state=42)
    shap_values = explainer.shap_values(X_sample)
    
    shap_sum = np.abs(shap_values).mean(axis=0)
    importance_df = pd.DataFrame({'feature': X_train_encoded.columns, 'shap_importance': shap_sum})
    features_to_keep = importance_df[importance_df['shap_importance'] > shap_threshold]['feature'].tolist()
    
    print(f"Features dropped by SHAP (pure noise): {len(cleaned_features) - len(features_to_keep)}")
    
    # Clean up massive objects from RAM
    del X_train_encoded, baseline_xgb, explainer, shap_values, X_sample, importance_df
    gc.collect()
    
    final_features = features_to_keep
    categorical_cols = [c for c in final_features if X_train_raw[c].dtype == 'object']
    numerical_cols = [c for c in final_features if X_train_raw[c].dtype != 'object']
    
    mlflow.log_metric("final_feature_count", len(final_features))


--- Executing SHAP Feature Selection Stage ---
Encoding temp data for baseline model...
Training baseline XGBoost for SHAP extraction...
Calculating SHAP values...
Features dropped by SHAP (pure noise): 198
🏃 View run XGBoost_SHAP_Feature_Selection at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/8/runs/f48fdd63b3af4759b19a121b5bed1701
🧪 View experiment at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/8


# Training

In [6]:
with mlflow.start_run(run_name="XGBoost_Training") as parent_run:
    print("\n--- Executing Training & Pipeline Construction ---")

    X_train_model = X_train_raw[final_features]
    X_val_model = X_val_raw[final_features]
    X_test_model = X_test_raw[final_features]

    # Preprocessors
    num_transformer = SimpleImputer(strategy='median')
    cat_transformer = ImbPipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='Missing_Category')),
        ('woe', ce.WOEEncoder())
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_transformer, numerical_cols),
            ('cat', cat_transformer, categorical_cols)
        ],
        remainder='drop' 
    )

    pipeline = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('undersampler', RandomUnderSampler(sampling_strategy=3/7, random_state=67)),
        ('classifier', xgb.XGBClassifier(scale_pos_weight=2.33, eval_metric="auc", random_state=67, n_jobs=-1))
    ])

    param_space = {
        'classifier__learning_rate': [0.05, 0.1],
        'classifier__n_estimators': [200, 400],
        'classifier__max_depth': [3, 5, 7],     
        'classifier__min_child_weight': [5, 15], 
        'classifier__gamma': [0.5, 2.0],          
        'classifier__subsample': [0.8],          
        'classifier__colsample_bytree': [0.8]    
    }

    all_combinations = list(ParameterGrid(param_space))
    sampled_combinations = random.sample(all_combinations, min(15, len(all_combinations)))

    best_val_auc = 0
    best_params = None
    best_pipeline = None

    print(f"Starting explicit grid search over {len(sampled_combinations)} combinations...")
    
    for i, params in enumerate(sampled_combinations):
        
        with mlflow.start_run(run_name=f"Manual_Search_Run_{i+1}", nested=True):
            clean_params = {k.replace('classifier__', ''): v for k, v in params.items()}
            mlflow.log_params(clean_params)
            
            pipeline.set_params(**params)
            pipeline.fit(X_train_model, y_train)
            
            y_train_pred = pipeline.predict_proba(X_train_model)[:, 1]
            y_val_pred = pipeline.predict_proba(X_val_model)[:, 1]
            
            train_auc = roc_auc_score(y_train, y_train_pred)
            val_auc = roc_auc_score(y_val, y_val_pred)
            
            mlflow.log_metrics({
                "train_roc_auc": train_auc,
                "val_roc_auc": val_auc
            })
            
            print(f"Run {i+1} | LR: {params['classifier__learning_rate']} | Est: {params['classifier__n_estimators']} | Depth: {params['classifier__max_depth']} | MinChild: {params['classifier__min_child_weight']} | Train: {train_auc:.4f} | Val: {val_auc:.4f}")


--- Executing Training & Pipeline Construction ---
Starting explicit grid search over 15 combinations...
Run 1 | LR: 0.05 | Est: 400 | Depth: 3 | MinChild: 15 | Train: 0.9123 | Val: 0.8992
🏃 View run Manual_Search_Run_1 at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/8/runs/fe4bee9d33c94dc68b40cc196e83af27
🧪 View experiment at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/8
Run 2 | LR: 0.1 | Est: 200 | Depth: 7 | MinChild: 15 | Train: 0.9630 | Val: 0.9180
🏃 View run Manual_Search_Run_2 at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/8/runs/b7b5fd097bcc41fc976f56c24e937a99
🧪 View experiment at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/8
Run 3 | LR: 0.05 | Est: 400 | Depth: 5 | MinChild: 15 | Train: 0.9419 | Val: 0.9133
🏃 View run Manual_Search_Run_3 at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/8/runs/7b66ad4b73d04

2026/05/03 13:44:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 13:44:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'XGBoost_Production_Model'.
2026/05/03 13:44:35 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost_Production_Model, version 1
Created version '1' of model 'XGBoost_Production_Model'.


Success! Winning Pipeline trained, evaluated, and saved to Model Registry.
🏃 View run XGBoost_Training at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/8/runs/0473866b3393415585e1aa887cd3fef3
🧪 View experiment at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/8
